# Project: Multi-Node Demand Forecaster & Inventory Optimizer
**Domain:** E-Commerce & Supply Chain Logistics  
**Dataset:** Olist Brazilian E-Commerce Public Dataset (100k+ Transactions)  

### **Business Objective**
To build a production-grade time-series forecasting engine that predicts daily product demand at the granular category level. The system is designed to optimize warehouse inventory management, prevent high-ticket stockouts, and reduce capital tied up in stagnant inventory.

### **Architectural Pipeline (Medallion Architecture)**
1. **Bronze Layer (Raw):** Ingestion of 9 highly fragmented, relational micro-service databases (orders, items, products, translations).
2. **Silver Layer (Spine):** Consolidation into a single, mathematically sound temporal spine. Includes upstream dictionary patching, semantic category consolidation (signal dilution repair), and strict data leakage amputation.
3. **Gold Layer (Matrix):** Mathematical compression of individual transactions into a daily time-series matrix, engineering target variables (`quantity`) and covariates (`freight_value`, `price`).
4. **Modeling (Future State):** Advanced Feature Engineering (Lags, Rolling Windows, Temporal encodings) feeding into an XGBoost regression model.

**Author:** Pruthvi   
**Date:** March 2026

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

In [ ]:
raw = pd.read_csv('olist_cleaned_silver_spine.csv', parse_dates=['order_purchase_timestamp', 'order_date'])
df=raw.copy()

There is one nan in **order_date** while it is created using the **order_purchase_timestamp** so the data entry may be corrupted  




In [ ]:
df.info()

In [ ]:
df.head(5)

In [ ]:
# Isolate the exact row where order_date became a NaN (NaT)
anomaly_row = df[df['order_date'].isnull()]

# Print the specific columns to investigate the corruption
print(anomaly_row[['order_id', 'order_purchase_timestamp', 'order_date']])

It is data loading or exporting proble so we will drop it


In [ ]:
# Amputate the corrupted CSV row
df = df.dropna(subset=['order_date']).copy()

We groupby the data so that we can calculate the **quantity,avg price, avg freight values** and we removed the columns **order_id	,customer_id	order_purchase_timestamp,	product_id**	because for daily demant forcasting we don't need it

In [ ]:
df1=df.groupby(['order_date', 'product_category_name_english']).agg(
    quantity=('order_id', 'count'),
    avg_price=('price', 'mean'),
    avg_freight=('freight_value', 'mean')
).reset_index()

In [ ]:
from statsmodels.tsa.stattools import adfuller

# Group the data to get the true daily heartbeat
daily_series = df1.groupby('order_date')['quantity'].sum()

# Run the ADF statistical test
result = adfuller(daily_series)

# Print the final score
print(f"ADF Statistic: {result[0]}")
print(f"p-value: {result[1]}")

In [ ]:
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose

# 1. We must set the date as the index for the math to work
df_daily = daily_series.to_frame() # Convert your grouped series back to a DataFrame
df_daily.index = pd.to_datetime(df_daily.index)

# 2. Run the decomposition (period=7 because e-commerce runs on a 7-day weekly cycle)
decomposition = seasonal_decompose(df_daily['quantity'], model='additive', period=7)

# 3. Plot the 3 hidden layers
fig = decomposition.plot()
fig.set_size_inches(15, 10)
plt.show()

In [ ]:
# 1. Amputate the toxic 2016 data by filtering the index
# We are forcing the timeline to start strictly on Jan 1, 2017
daily_series_clean = daily_series[daily_series.index >= '2017-01-01'].copy()

# 2. Print the confirmation
print(f"Original Timeline Start: {daily_series.index.min().date()}")
print(f"Cleaned Timeline Start: {daily_series_clean.index.min().date()}")
print(f"Total rows removed: {len(daily_series) - len(daily_series_clean)}")

There is no null values and all data types are perfact

In [ ]:
df1.info()

In [ ]:
df1=df1.sort_values(by='order_date')

split_index=int(len(df1)*0.8)
train=df1.iloc[:split_index]
test=df1.iloc[split_index:]

x_train=train.drop(columns=['quantity'])
y_train=train['quantity']

x_test=test.drop(columns=['quantity'])
y_test=test['quantity']

print(f"Training data spans from: {train['order_date'].min()} to {train['order_date'].max()}")
print(f"Testing data spans from: {test['order_date'].min()} to {test['order_date'].max()}")
print(f"X_train shape: {x_train.shape}, X_test shape: {x_test.shape}")


In [ ]:
x_train